# Entrenamiento YOLOv8 — Detección de Aves

**Proyecto:** Visión Artificial — Reconocimiento de Aves con YOLO

Corre este notebook en **Google Colab** con GPU activada
(`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`).

Corre las celdas **en orden, de arriba hacia abajo**, sin saltarte ninguna.

## 1. Instalación de dependencias

In [ ]:
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()

## 2. Descarga del dataset (Roboflow)

Dataset: **"birds" by kurisnis** — https://universe.roboflow.com/kurisnis/birds-olyak

> **Importante:** usamos la **versión 3** del dataset (665 imágenes), no la versión 1
> (que solo tiene 6 imágenes de prueba y no sirve para entrenar).

Pega tu API key de Roboflow donde dice `YOUR_API_KEY`
(la obtienes en https://app.roboflow.com/settings/api).

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("kurisnis").project("birds-olyak")
version = project.version(3)
dataset = version.download("yolov8")

print("Dataset descargado en:", dataset.location)

## 3. Verificar el dataset

Confirmamos que existan los splits `train`, `valid` y `test`, y revisamos
el `data.yaml` y cuántas imágenes tiene cada split.

In [ ]:
import os

print("Contenido del dataset:", os.listdir(dataset.location))
print()

with open(f"{dataset.location}/data.yaml", "r") as f:
    print(f.read())

for split in ["train", "valid", "test"]:
    ruta = f"{dataset.location}/{split}/images"
    if os.path.exists(ruta):
        print(f"{split}: {len(os.listdir(ruta))} imágenes")
    else:
        print(f"{split}: NO EXISTE")

## 4. Entrenamiento del modelo

Usamos `yolov8n.pt` (versión *nano*): entrena rápido en la GPU gratuita de Colab
y es suficiente para este dataset de una sola clase.

Los resultados se guardan en una carpeta fija (`/content/runs_aves_final`) para
no perdernos entre corridas como pasó antes.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project="/content/runs_aves_final",
    name="entrenamiento",
    exist_ok=True,
    seed=42
)

# Ruta fija y predecible al mejor modelo entrenado
RUTA_BEST = "/content/runs_aves_final/entrenamiento/weights/best.pt"
print("\nModelo guardado en:", RUTA_BEST)
print("¿Existe?", os.path.exists(RUTA_BEST))

## 5. Evaluación del modelo

Métricas sobre el set de validación: **mAP50**, **mAP50-95**, **precisión** y **recall**.

In [ ]:
best_model = YOLO(RUTA_BEST)

metrics = best_model.val(data=f"{dataset.location}/data.yaml")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precisión:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 6. Inferencia sobre imágenes de prueba (generación de evidencias)

Corremos el modelo entrenado sobre el set de `test/` para generar imágenes
con bounding boxes. Estas son las que se suben a la carpeta `evidencias/`
del repositorio.

In [ ]:
import glob

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
if len(test_images) == 0:
    # Si no hay test, usamos imágenes de valid como respaldo
    test_images = glob.glob(f"{dataset.location}/valid/images/*.jpg")

print(f"Imágenes de prueba encontradas: {len(test_images)}")

pred_results = best_model.predict(
    source=test_images[:15],
    conf=0.35,
    save=True,
    project="/content/runs_aves_final",
    name="predicciones_evidencias",
    exist_ok=True
)

print("Predicciones guardadas en: /content/runs_aves_final/predicciones_evidencias/")

### Visualizar algunas predicciones directamente en el notebook

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pred_imgs = glob.glob("/content/runs_aves_final/predicciones_evidencias/*.jpg")[:6]

if len(pred_imgs) == 0:
    print("No se encontraron imágenes de predicción para mostrar.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten() if len(pred_imgs) > 1 else [axes]
    for ax, img_path in zip(axes, pred_imgs):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.axis("off")
    for ax in axes[len(pred_imgs):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("/content/grid_evidencias.png", dpi=150)
    plt.show()

## 7. Descargar resultados para subir al repositorio

Esta celda empaqueta en un solo ZIP todo lo que necesitas para la carpeta
`evidencias/` de tu repositorio: las imágenes con detecciones y la gráfica
de curvas de entrenamiento.

In [ ]:
import shutil

# Copiamos el modelo entrenado con nombre claro
shutil.copy(RUTA_BEST, "/content/modelo_aves_best.pt")

# Armamos una carpeta limpia con todo lo necesario para evidencias/
os.makedirs("/content/evidencias_para_subir", exist_ok=True)

# Imágenes con bounding boxes
shutil.copytree(
    "/content/runs_aves_final/predicciones_evidencias",
    "/content/evidencias_para_subir/predicciones",
    dirs_exist_ok=True
)

# Gráfica de curvas de entrenamiento (loss, mAP, precisión, recall)
ruta_results_png = "/content/runs_aves_final/entrenamiento/results.png"
if os.path.exists(ruta_results_png):
    shutil.copy(ruta_results_png, "/content/evidencias_para_subir/results.png")

# Comprimimos todo en un ZIP fácil de descargar
shutil.make_archive("/content/evidencias_para_subir", "zip", "/content/evidencias_para_subir")

print("Listo. Descarga estos archivos:")
print("  /content/evidencias_para_subir.zip  -> descomprimir dentro de evidencias/ en tu repo")
print("  /content/modelo_aves_best.pt        -> (opcional) modelo entrenado")

In [ ]:
from google.colab import files

files.download("/content/evidencias_para_subir.zip")
files.download("/content/modelo_aves_best.pt")

## 8. Siguiente paso

1. Descomprime `evidencias_para_subir.zip` y copia su contenido
   (`predicciones/` y `results.png`) dentro de la carpeta `evidencias/`
   de tu repositorio.
2. Anota los valores de mAP50, mAP50-95, precisión y recall (los imprimió
   la celda 5) en la tabla de resultados del README.md.
3. Haz commit y push de todo al repositorio de GitHub.